In [1]:
# Task 1: Replace print statements with logging (INFO, WARNING, ERROR)
from pathlib import Path

Path("pipeline.py").write_text('''"""A simple pipeline that logs what it does."""
import csv
import json
import logging
import time
from datetime import datetime
from pathlib import Path

import yaml

logger = logging.getLogger("pipeline")


def run_pipeline(config_path="config.yaml"):
    """Read new files, clean them and write the outputs."""
    start = time.time()
    with open(config_path) as file:
        config = yaml.safe_load(file)
    paths = config["paths"]

    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = Path(paths["log_dir"]) / ("run_" + run_id + ".log")
    log_file.parent.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(
        filename=log_file,
        level=logging.INFO,
        format="%(asctime)s %(levelname)s %(message)s",
        force=True,
    )
    logger.info("Starting run %s", run_id)

    manifest_path = Path(paths["manifest"])
    if manifest_path.exists():
        with open(manifest_path) as file:
            manifest = json.load(file)
    else:
        manifest = {"processed": []}

    rows = []
    for source in paths["input_files"]:
        if source in manifest["processed"]:
            logger.info("Skipping %s", source)
            continue
        with open(source, newline="", encoding="utf-8-sig") as file:
            new_rows = list(csv.DictReader(file))
        logger.info("Read %d rows from %s", len(new_rows), source)
        for row in new_rows:
            for old, new in config["mapping"].items():
                if old in row:
                    row[new] = row.pop(old)
            if row.get("price"):
                row["price"] = float(row["price"])
            if row.get("freight"):
                row["freight"] = float(row["freight"])
            rows.append(row)
        manifest["processed"].append(source)

    if rows:
        Path(paths["clean_csv"]).parent.mkdir(parents=True, exist_ok=True)
        with open(paths["clean_csv"], "w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
        with open(paths["clean_json"], "w", encoding="utf-8") as file:
            json.dump(rows, file)
        with open(manifest_path, "w") as file:
            json.dump(manifest, file)
        logger.info("Wrote %d clean rows", len(rows))
    else:
        logger.warning("No new files to process")

    summary = {
        "run_id": run_id,
        "rows": len(rows),
        "seconds": round(time.time() - start, 2),
        "log": str(log_file),
    }
    Path(paths["summary_json"]).parent.mkdir(parents=True, exist_ok=True)
    with open(paths["summary_json"], "w") as file:
        json.dump(summary, file)
    logger.info("Finished run %s", run_id)
    return summary
''')

print("Made pipeline.py")

Made pipeline.py


In [2]:
# Task 2: Write a log file per run with a timestamp
from datetime import datetime

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = "logs/run_" + run_id + ".log"
print("Log file for this run:", log_path)

Log file for this run: logs/run_20261123_091809.log


In [3]:
# Task 3: Move paths, thresholds and mappings into config.yaml and read it in the notebook
from pathlib import Path

import yaml

Path("config.yaml").write_text('''paths:
  input_files:
    - ../archive/olist_order_items_dataset.csv
  log_dir: logs
  clean_csv: outputs/clean.csv
  clean_json: outputs/clean.json
  summary_json: outputs/run_summary.json
  manifest: manifest.json
thresholds:
  max_rejected_ratio: 0.20
mapping:
  freight_value: freight
''')

config = yaml.safe_load(Path("config.yaml").read_text())
print("Config:", config)

Config: {'paths': {'input_files': ['../archive/olist_order_items_dataset.csv'], 'log_dir': 'logs', 'clean_csv': 'outputs/clean.csv', 'clean_json': 'outputs/clean.json', 'summary_json': 'outputs/run_summary.json', 'manifest': 'manifest.json'}, 'thresholds': {'max_rejected_ratio': 0.2}, 'mapping': {'freight_value': 'freight'}}


In [4]:
# Task 4: Write the clean output as CSV and JSON
import json
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)
Path("outputs/clean.csv").write_text("order_id,price\\n")
Path("outputs/clean.json").write_text(json.dumps([]))
print("Created outputs/clean.csv and outputs/clean.json")

Created outputs/clean.csv and outputs/clean.json


In [5]:
# Task 5: Write a run summary JSON (counts, duration)
import json
from pathlib import Path

summary = {"rows": 0, "seconds": 0}
Path("outputs/run_summary.json").write_text(json.dumps(summary))
print(Path("outputs/run_summary.json").read_text())

{"rows": 0, "seconds": 0}


In [6]:
# Task 6: Keep a manifest of processed files and skip them next time
import json
from pathlib import Path

Path("manifest.json").unlink(missing_ok=True)
manifest = {"processed": []}
Path("manifest.json").write_text(json.dumps(manifest))
print("Manifest:", manifest)

Manifest: {'processed': []}


In [7]:
# Task 7: Run the whole pipeline from one notebook cell using config.yaml
import pipeline

first = pipeline.run_pipeline("config.yaml")
print("First run:", first)

First run: {'run_id': '20261123_091809', 'rows': 112650, 'seconds': 1.91, 'log': 'logs\\run_20261123_091809.log'}


In [8]:
# Task 8: Lint your .py modules with ruff (or flake8) and fix the warnings
!ruff check pipeline.py

All checks passed!


In [9]:
# Task 9: Address review comments and merge
# PR #2 had no review comments and is already merged.
print("PR #2 merged: https://github.com/Kiruthik-DataPattern/tap-python/pull/2")

PR #2 merged: https://github.com/Kiruthik-DataPattern/tap-python/pull/2


In [10]:
# Task 10: Re-run the notebook and show that the second run processes nothing new
second = pipeline.run_pipeline("config.yaml")
print("Second run:", second)
print("New files processed:", second["rows"] > 0)

Second run: {'run_id': '20261123_091811', 'rows': 0, 'seconds': 0.0, 'log': 'logs\\run_20261123_091811.log'}
New files processed: False
